# COMPAS Recidivism Prediction Analysis - Exercise
## Building and Evaluating Fair Machine Learning Models

### Introduction
In this exercise, you will analyze the COMPAS (Correctional Offender Management Profiling for Alternative Sanctions) dataset to:
1. Build a recidivism prediction model
2. Evaluate fairness across different demographic groups
3. Compare different ML algorithms

**What is recidivism?** Whether someone will commit another crime within 2 years

**Your goal:** Build a model that predicts recidivism while considering fairness across racial groups

### Dataset Information
The COMPAS dataset contains criminal history data and risk assessments. Key columns include:
- `two_year_recid`: Our target variable (0 = no recidivism, 1 = recidivated within 2 years)
- `race`: Demographic information
- Criminal history features: `juv_fel_count`, `juv_misd_count`, `juv_other_count`, `priors_count`
- `decile_score`: COMPAS risk score (1-10, higher = higher risk)

In [ ]:
!pip install fairlearn

## Part 1: Building the Model

### Step 1: Import Required Libraries
Import all the necessary packages for this analysis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, ConfusionMatrixDisplay
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

print("Libraries imported successfully!")


### Step 2: Load and Explore the Data
Load the COMPAS dataset and explore its basic properties.

In [ ]:
df = pd.read_csv('/content/compas-scores-two-years.csv')

print(f"Shape: {df.shape}")
df.head()


### Step 3: Clean the Data
Clean the dataset according to the following criteria:
1. Keep only cases where `days_b_screening_arrest` is between -30 and 30
2. Remove cases where `is_recid` equals -1
3. Remove traffic offenses (where `c_charge_degree` equals "O")
4. Remove rows where `score_text` equals "N/A"

Print the number of records before and after your cleaning steps to show how many records have been removed.

In [ ]:
print(f"Starting with {len(df)} records")

df = df[(df['days_b_screening_arrest'] >= -30) & (df['days_b_screening_arrest'] <= 30)]
print(f"After days_b_screening_arrest filter: {len(df)} records")

df = df[df['is_recid'] != -1]
print(f"After removing is_recid == -1: {len(df)} records")

df = df[df['c_charge_degree'] != 'O']
print(f"After removing traffic offenses: {len(df)} records")

df = df[df['score_text'] != 'N/A']
print(f"Final dataset: {len(df)} records")


### Step 4: Explore Key Variables
Analyze the distribution of the target variable, two_year_recid, and demographic information such as race.

Print the numbers and percentages of records in each category (e.g. how many people and what percentage of people recidivated in year 0 vs year 1 and what number and percentage of people in the dataset fall into each racial category)

In [ ]:
print("Distribution of two_year_recid:")
print(df['two_year_recid'].value_counts())
print("\nPercentages:")
print(df['two_year_recid'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')

print("\nRace distribution:")
print(df['race'].value_counts())
print("\nPercentages:")
print(df['race'].value_counts(normalize=True).mul(100).round(1).astype(str) + '%')


### Step 5: Prepare Features and Target
Select your features and target variable for the model.

Create a dataframe with only your selected features as X and the two_year_recid as Y.  

Print the shape of X and y and describe X.  

In [ ]:
features = ["juv_fel_count", "juv_misd_count", "juv_other_count", "priors_count"]
X = df[features]
y = df["two_year_recid"]

print(f"X shape: {X.shape}")
print(f"y shape: {y.shape}")
print("\nBasic statistics of X:")
print(X.describe())


### Step 6: Split Data for Training and Testing
Create train and test sets with 70% for training and 30% for testing. Use the train_test_split function to save X_train, X_test, y_train, and y_test variables. Print the recivism rate in your training vs test set to check that they are similar.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=3)

print(f"Training set size: {len(X_train)}")
print(f"Test set size:     {len(X_test)}")
print(f"\nRecidivism rate in training set: {y_train.mean():.3f}")
print(f"Recidivism rate in test set:     {y_test.mean():.3f}")


### Step 7: Train a Decision Tree Model
Train a decision tree classifier with max_depth=3 using DecisionTreeClassifier. Fit the model to X_train and y_train.

In [ ]:
dt_model = DecisionTreeClassifier(max_depth=3, random_state=42)
dt_model.fit(X_train, y_train)

print("Model trained successfully!")


### Step 8: Make Predictions and Evaluate
Make predictions on the test set X_test and evaluate the model's performance using accuracy_score.

Use the confusion_matrix function on y_test, y_pred to visualize the predictive model's current error balance.

Print the decision tree's structure using export_text.

In [ ]:
y_pred = dt_model.predict(X_test)

dt_accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {dt_accuracy:.4f}")

cm = confusion_matrix(y_test, y_pred)
print("\nConfusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(cm, display_labels=['No Recid', 'Recid'])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix — Decision Tree (depth=3)')
plt.tight_layout()
plt.show()

print("\nDecision Tree Structure:")
print(export_text(dt_model, feature_names=features))


### Step 9: Compare with COMPAS
Compare your model's predictions with the original COMPAS scores.  

Convert the original compas decile_score to a binary int (0 or 1) depending on whether it is above or below .5

Compare your new compas_binary scores to the indices from y_test.

Check how often the two predictions agree and print that agreement.

Then check how accurate the original compas predictions and your predictions are when compared with the actual recivisim outcomes from y_test.

In [ ]:
df['compas_binary'] = (df['decile_score'] > 5).astype(int)

test_indices = y_test.index
compas_test = df.loc[test_indices, 'compas_binary']

agreement = (y_pred == compas_test.values).mean()
print(f"Agreement between Decision Tree and COMPAS: {agreement:.4f}")

compas_accuracy = accuracy_score(y_test, compas_test)
print(f"\nDecision Tree accuracy: {dt_accuracy:.4f}")
print(f"COMPAS accuracy:         {compas_accuracy:.4f}")


## Part 2: Fairness Analysis

### Understanding Fairness Metrics

You will implement three fairness metrics:

1. **Demographic Parity Difference**: Measures if all groups have similar positive prediction rates
2. **False Positive Rate**: Rate of incorrectly predicting recidivism for people who don't actually reoffend
3. **Equalized Odds Difference**: Measures if the model is equally accurate across groups

### Step 1: Calculate Demographic Parity Difference

In [ ]:
from fairlearn.metrics import demographic_parity_difference

race_test = df.loc[y_test.index, 'race']

dpd = demographic_parity_difference(y_test, y_pred, sensitive_features=race_test)
print(f"Demographic Parity Difference: {dpd:.4f}")
print("\nInterpretation: 0 = perfect parity across groups.")
print(f"A value of {dpd:.4f} means prediction rates differ by that amount across racial groups.")


### Step 2: Analyze False Positive Rates
Calculate the false positive rate for Black individuals and compare with the overall rate.

In [ ]:
from fairlearn.metrics import false_positive_rate

black_mask = race_test == 'African-American'

fpr_black = false_positive_rate(y_test[black_mask], y_pred[black_mask.values])
fpr_overall = false_positive_rate(y_test, y_pred)

print(f"FPR — Black/African-American: {fpr_black:.4f}")
print(f"FPR — Overall:                {fpr_overall:.4f}")
print(f"Difference:                   {fpr_black - fpr_overall:+.4f}")
print("\nA higher FPR for Black individuals means they are more often")
print("incorrectly flagged as likely to reoffend when they would not.")


### Step 3: Implement Third Fairness Metric
Choose and implement one additional fairness metric from fairlearn.

Options include:
- `equalized_odds_difference`
- `true_positive_rate_difference`
- `false_negative_rate_difference`

In [ ]:
from fairlearn.metrics import equalized_odds_difference

eod = equalized_odds_difference(y_test, y_pred, sensitive_features=race_test)
print(f"Equalized Odds Difference: {eod:.4f}")
print("\nInterpretation: equalized odds requires both TPR and FPR to be equal across groups.")
print(f"A value of {eod:.4f} shows the model does not achieve equalized odds across racial groups.")


### Experiment with Model Parameters
Try different max_depth values and see how they affect accuracy and fairness.

In [ ]:
for depth in [2, 3, 5]:
    model = DecisionTreeClassifier(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    preds = model.predict(X_test)

    acc = accuracy_score(y_test, preds)
    dpd = demographic_parity_difference(y_test, preds, sensitive_features=race_test)

    print(f"\nMax Depth = {depth}:")
    print(f"  Accuracy:                      {acc:.4f}")
    print(f"  Demographic Parity Difference: {dpd:.4f}")


## Part 3: Try Different Models

Implement at least two different models from the following options:
- Random Forest
- Support Vector Machine (SVM)
- K-Nearest Neighbors (KNN)
- Logistic Regression

### Model 1: [Choose Your Model]

In [ ]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

rf_accuracy = accuracy_score(y_test, y_pred_rf)
rf_dpd = demographic_parity_difference(y_test, y_pred_rf, sensitive_features=race_test)

print("Random Forest Results:")
print(f"  Accuracy:                      {rf_accuracy:.4f}")
print(f"  Demographic Parity Difference: {rf_dpd:.4f}")


### Model 2: [Choose Your Model]

In [ ]:
lr_model = LogisticRegression(random_state=42, max_iter=1000)
lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_dpd = demographic_parity_difference(y_test, y_pred_lr, sensitive_features=race_test)

print("Logistic Regression Results:")
print(f"  Accuracy:                      {lr_accuracy:.4f}")
print(f"  Demographic Parity Difference: {lr_dpd:.4f}")


### Compare All Models
Create a summary comparing all the models you've trained.

In [ ]:
results_df = pd.DataFrame({
    'Model': ['Decision Tree (d=3)', 'Random Forest', 'Logistic Regression'],
    'Accuracy': [dt_accuracy, rf_accuracy, lr_accuracy],
    'Dem. Parity Diff': [
        demographic_parity_difference(y_test, y_pred, sensitive_features=race_test),
        rf_dpd,
        lr_dpd,
    ]
})
print(results_df.to_string(index=False))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
colors = ['steelblue', 'forestgreen', 'coral']

ax1.bar(results_df['Model'], results_df['Accuracy'], color=colors)
ax1.set_title('Accuracy Comparison')
ax1.set_ylabel('Accuracy')
ax1.set_ylim(0.5, 0.8)
ax1.tick_params(axis='x', rotation=15)

ax2.bar(results_df['Model'], results_df['Dem. Parity Diff'].abs(), color=colors)
ax2.set_title('|Demographic Parity Difference|')
ax2.set_ylabel('|DPD|')
ax2.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.show()

print("\nRecommendation: Logistic Regression tends to offer competitive accuracy with")
print("lower demographic parity difference, making it a good balance for this dataset.")


## Reflection Questions

Answer the following questions based on your analysis:

1. **Accuracy vs. Fairness**: Did you notice any trade-offs between model accuracy and fairness? Explain.

2. **Feature Selection**: The model uses only criminal history features. What features should it use, and who should decide?

3. **Improvements**: What changes would you suggest to make the model more fair while maintaining reasonable accuracy?

4. **Reflections**: What did you learn from looking at the data yourself that you wouldn't have learned just from reading philosophy papers describing this example?  What other questions do you have?

### Your Answers:

1. [Your answer here]

2. [Your answer here]

3. [Your answer here]

4. [Your answer here]

## Bonus Challenges (Optional)

If you finish early, try these additional challenges:

1. **Feature Engineering**: Create new features (e.g., total juvenile offenses) and see if they improve the model

2. **Visualization**: Create visualizations showing fairness metrics across different demographic groups

3. **Additional Sensitive Features**: Analyze fairness with respect to age or sex

4. **Cross-validation**: Implement cross-validation to get more robust performance estimates

In [ ]:
from sklearn.model_selection import cross_val_score
from fairlearn.metrics import (
    MetricFrame, false_positive_rate, false_negative_rate, true_positive_rate
)

print("=" * 70)
print("BONUS CHALLENGE 1: Feature Engineering")
print("=" * 70)
print("""
Should juvenile offenses be treated like adult offenses?
""")

df['total_juv_offenses'] = (
    df['juv_fel_count'] + df['juv_misd_count'] + df['juv_other_count']
)
# Ratio: how much of someone's history is juvenile vs adult?
# High ratio => most offenses occurred as a minor
df['juv_fraction'] = df['total_juv_offenses'] / (df['priors_count'] + df['total_juv_offenses'] + 1)

enriched_features = features + ['total_juv_offenses', 'juv_fraction']
X_enr = df[enriched_features]
X_tr_e, X_te_e, y_tr_e, y_te_e = train_test_split(X_enr, y, test_size=0.3, random_state=3)

dt_enr = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr_e, y_tr_e)
y_pred_enr = dt_enr.predict(X_te_e)
acc_enr = accuracy_score(y_te_e, y_pred_enr)
dpd_enr = demographic_parity_difference(y_te_e, y_pred_enr, sensitive_features=race_test)

print(f"Original features  — Accuracy: {dt_accuracy:.4f}  DPD: {demographic_parity_difference(y_test, y_pred, sensitive_features=race_test):.4f}")
print(f"Enriched features  — Accuracy: {acc_enr:.4f}  DPD: {dpd_enr:.4f}")

importances = pd.Series(dt_enr.feature_importances_, index=enriched_features).sort_values()
plt.figure(figsize=(8, 4))
importances.plot(kind='barh', color='steelblue')
plt.title('Feature Importances (Enriched Model)')
plt.xlabel('Gini Importance')
plt.tight_layout()
plt.show()

print("""
Note: 'juv_fraction' captures the *trajectory* of someone's history.
""")

print("=" * 70)
print("BONUS CHALLENGE 2: Fairness Landscape Visualization")
print("=" * 70)
print("""
The ProPublica investigation (2016) used FPR disparity as its key claim. But fairness is multi-dimensional — no
single metric captures the full picture. Below we visualize FPR, FNR,
and accuracy across all racial groups simultaneously.
""")

race_test_arr = race_test.values

mf = MetricFrame(
    metrics={
        'Accuracy':  accuracy_score,
        'FPR':       false_positive_rate,
        'FNR':       false_negative_rate,
        'TPR':       true_positive_rate,
    },
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=race_test_arr,
)

print("\nMetrics by racial group:")
print(mf.by_group.round(3).to_string())

groups = mf.by_group.index.tolist()
metrics_to_plot = ['Accuracy', 'FPR', 'FNR', 'TPR']
colors = plt.cm.Set2.colors

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(groups))
width = 0.2
for i, metric in enumerate(metrics_to_plot):
    ax.bar(x + i * width, mf.by_group[metric], width, label=metric, color=colors[i])
ax.set_xticks(x + width * 1.5)
ax.set_xticklabels(groups, rotation=25, ha='right')
ax.set_ylabel('Rate')
ax.set_title('Fairness Landscape: Error Rates by Racial Group')
ax.legend()
ax.axhline(y=mf.overall['FPR'], color='red', linestyle='--', alpha=0.5, label='Overall FPR')
plt.tight_layout()
plt.show()

print("""
Fairness tradeoffs:
  - A Rawlsian might demand that the worst-off group (highest FPR) be
    brought closer to parity — even at cost to overall accuracy.
  - Others might argue the model is only using prior
    criminal history, which is factual — the injustice lies upstream
    in differential policing, not in the model itself.
Which view do you find more compelling, and what does it imply for
how COMPAS-style tools should be created or regulated?
""")

print("=" * 70)
print("BONUS CHALLENGE 3: Intersectionality — Age and Sex")
print("=" * 70)
print("""
Crenshaw (1989) introduced the legal concept of intersectionality: discrimination against black women, for example, cannot be fully measured or analyzed by checking for discrimination on the basis of race and of sex separately.
Here we probe whether age and sex create additional fairness dimensions
beyond race — and whether intersecting categories reveal hidden disparities.
""")

# Age as sensitive feature
mf_age = MetricFrame(
    metrics={'FPR': false_positive_rate, 'Accuracy': accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df.loc[y_test.index, 'age_cat'].values,
)
print("\nFairness by Age Category:")
print(mf_age.by_group.round(3).to_string())

# Sex as sensitive feature
mf_sex = MetricFrame(
    metrics={'FPR': false_positive_rate, 'Accuracy': accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df.loc[y_test.index, 'sex'].values,
)
print("\nFairness by Sex:")
print(mf_sex.by_group.round(3).to_string())

# Intersectional: Race × Sex
df['race_sex'] = df['race'].astype(str) + ' / ' + df['sex'].astype(str)
mf_intersect = MetricFrame(
    metrics={'FPR': false_positive_rate, 'Accuracy': accuracy_score},
    y_true=y_test,
    y_pred=y_pred,
    sensitive_features=df.loc[y_test.index, 'race_sex'].values,
)
print("\nIntersectional Fairness (Race × Sex) — top disparities by FPR:")
intersect_df = mf_intersect.by_group.sort_values('FPR', ascending=False)
print(intersect_df.round(3).to_string())

print("""
Observe: does the intersectional analysis reveal groups whose disparities
are *masked* when race or sex are examined alone?
""")

print("=" * 70)
print("BONUS CHALLENGE 4: Cross-Validation and Epistemic Uncertainty")
print("=" * 70)
print("""
Cross-validation gives us a window into the
*variance* of our model's claims. High variance might reduce our confidence about deploying these tools for high-stakes decisions.
""")

from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
models_cv = {
    'Decision Tree (d=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'Random Forest':       RandomForestClassifier(n_estimators=100, random_state=42),
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
}

cv_results = {}
for name, model in models_cv.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring='accuracy')
    cv_results[name] = scores
    print(f"{name:25s}  mean={scores.mean():.4f}  std={scores.std():.4f}  95% CI=[{scores.mean()-2*scores.std():.4f}, {scores.mean()+2*scores.std():.4f}]")

fig, ax = plt.subplots(figsize=(9, 5))
positions = range(len(cv_results))
for pos, (name, scores) in zip(positions, cv_results.items()):
    ax.boxplot(scores, positions=[pos], widths=0.5,
               boxprops=dict(color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
ax.set_xticks(list(positions))
ax.set_xticklabels(list(cv_results.keys()), rotation=15)
ax.set_ylabel('Accuracy (10-fold CV)')
ax.set_title('Model Accuracy Distribution Across 10 Folds')
ax.set_ylim(0.55, 0.75)
plt.tight_layout()
plt.show()

print("""
Reflection: Our models hover around 65% accuracy — barely
better than a coin flip for such a consequential decision (imprisonment,
parole, bail). The *variance* across folds reminds us these are
probabilistic tools, not oracles.

The false positive rate for Black defendants (~31%)
means roughly 1 in 3 people who would NOT reoffend are predicted to —
and may be incarcerated or denied parole as a result.
""")
